# Evaluate Exposure of WRs

<br><br/>
#### *Initialize Environment*

In [ ]:
import sys
from pathlib import Path

# Identify Project Root
PROJECT_ROOT = next(
    parent
    for parent in [Path.cwd(), *Path.cwd().parents]
    if (parent / "pyproject.toml").exists()
)
sys.path.insert(0, str(PROJECT_ROOT))

# Import Shared Project Paths
from utils.paths import (
    CLEAN_DATA_DIR,
)

import polars as pl 
import pandas as pd 
import numpy as np
import plotly.express as px
import plotly.io as pio
pio.renderers.default = "png"

# Import schema_map
import sys
from pathlib import Path
# Identify Project Root

PROJECT_ROOT = next(

    parent

    for parent in [Path.cwd(), *Path.cwd().parents]

    if (parent / "pyproject.toml").exists()

)



# Import Shared Data Schemas

sys.path.insert(0, str(PROJECT_ROOT))

from utils.schema import schema_map

In [ ]:
# ==============================
# Specify Position of Interest
# ==============================

position_of_interest = "WR"

<br><br/>
## Load Data
***

In [ ]:
# =============================
# Players Data
# =============================

players_df = (
    pl.read_csv(
        str(CLEAN_DATA_DIR / "players.csv"), 
        schema_overrides = schema_map["players"]
    )
)

# -----------------------------
# Only Keep Players of Interest
# -----------------------------
players_df = (
    players_df
    .filter(pl.col("nfl_position") == position_of_interest)
)

# -----------------------------
# Only Keep Variables of Interest
# -----------------------------
players_df = (
    players_df
    .select(
        "nfl_id", 
        "display_name", 
        "draft_year", 
        "nfl_position"
    )
)

# -----------------------------
# Identify Players of Interest
# -----------------------------

players_of_interest = (
    players_df
    .select("nfl_id")
    .unique()
    .to_series()
    .to_list()    
)

print("Players Data:")
print(f"    - Number of Players In Position Of Interest From Rookie Cohort: {len(players_of_interest)}")
players_df

In [ ]:
# =============================
# Combine Results Data
# =============================

combine_results_df = (
    pl.read_csv(
        str(CLEAN_DATA_DIR / "combine_results.csv"), 
        schema_overrides = schema_map["combine_results"]
    )
)

# -----------------------------
# Only Keep Players of Interest
# -----------------------------

combine_results_df = (
    combine_results_df
    .filter(pl.col("nfl_id").is_in(players_of_interest))
)

# -----------------------------
# Drop Unneeded Columns
# -----------------------------

combine_results_df = (
    combine_results_df
    .drop("draft_year")
)

print("Combine Results Data:")
print(f"    - Number of Players In This Dataset: {combine_results_df.select(pl.col("nfl_id")).n_unique()}")
combine_results_df

In [ ]:
# =============================
# Combine Tracking Data
# =============================

combine_tracking_df = (
    pl.read_csv(
        str(CLEAN_DATA_DIR / "combine_tracking.csv"), 
        schema_overrides = schema_map["combine_tracking"]
    )
)

# -----------------------------
# Only Keep Players of Interest
# -----------------------------

combine_tracking_df = (
    combine_tracking_df
    .filter(pl.col("nfl_id").is_in(players_of_interest))
)

# -----------------------------
# Don't Keep Ball Tracking Data
# -----------------------------

combine_tracking_df = (
    combine_tracking_df
    .filter(pl.col("entity_type") == "PLAYER")
)

print("Combine Tracking Data:")
print(f"    - Number of Players In This Dataset: {combine_tracking_df.select(pl.col("nfl_id")).n_unique()}")
combine_tracking_df

In [ ]:
# =============================
# Player Exposure Data
# =============================

player_exposure_df = (
    pl.read_csv(
        str(CLEAN_DATA_DIR / "player_career_successes.csv"), 
        schema_overrides = schema_map["player_career_successes"]
    )
)

# -----------------------------
# Only Keep Players of Interest
# -----------------------------

player_exposure_df = (
    player_exposure_df
    .filter(pl.col("nfl_id").is_in(players_of_interest))
)

# -----------------------------
# Only Keep Exposure Variables
# -----------------------------

player_exposure_df = (
    player_exposure_df
    .select(
        "nfl_id",
        "career_offensive_snaps", 
        "career_defensive_snaps",
        "career_special_teams_snaps", 
        "career_games_active"
    )
    # ------------------------------------
    # Add variable for total snaps played
    # ------------------------------------
    .with_columns(
        career_total_snaps = (
            pl.sum_horizontal(
                "career_offensive_snaps", 
                "career_defensive_snaps", 
                "career_special_teams_snaps"
            )
        )
    )
)

print("Player Exposure Data:")
print(f"    - Number of Players In This Dataset: {player_exposure_df.select(pl.col("nfl_id")).n_unique()}")
player_exposure_df

In [ ]:
# =============================
# Game Tracking Data
# =============================

# -----------------------------
# 2023 Game Tracking Data
# -----------------------------

game_tracking_2023_df = (
    pl.read_csv(
        str(CLEAN_DATA_DIR / "game_tracking_2023.csv"), 
        schema_overrides = schema_map["game_tracking"]
    )

    # ----------------------------------
    # Only Keep Players of Interest
    # ----------------------------------
    .filter(pl.col("nfl_id").is_in(players_of_interest))
)

# -----------------------------
# 2024 Game Tracking Data
# -----------------------------

game_tracking_2024_df = (
    pl.read_csv(
        str(CLEAN_DATA_DIR / "game_tracking_2024.csv"), 
        schema_overrides = schema_map["game_tracking"]
    )

    # ----------------------------------
    # Only Keep Players of Interest
    # ----------------------------------
    .filter(pl.col("nfl_id").is_in(players_of_interest))
)

# -----------------------------
# 2025 Game Tracking Data
# -----------------------------

game_tracking_2025_df = (
    pl.read_csv(
        str(CLEAN_DATA_DIR / "game_tracking_2025.csv"), 
        schema_overrides = schema_map["game_tracking"]
    )

    # ----------------------------------
    # Only Keep Players of Interest
    # ----------------------------------
    .filter(pl.col("nfl_id").is_in(players_of_interest))
)

# -----------------------------
# Combine All Tracking Datasets
# -----------------------------

game_tracking_df = (
    pl.concat(
        [
            game_tracking_2023_df, 
            game_tracking_2024_df, 
            game_tracking_2025_df
        ]
    )
)

print("Game Tracking Data:")
print(f"    - Number of Players In This Dataset: {game_tracking_df.select(pl.col("nfl_id")).n_unique()}")
print(f"    - Number of Players Missing: {len(players_of_interest) - game_tracking_df.select(pl.col("nfl_id")).n_unique()}")
game_tracking_df

<br><br/>
## Drill-Specific Exposure
***

#### Among Players of Interest, How Many Had Data For Official Drill Times?

In [ ]:
# ==================================
# Find Percentage of Players of Interest
# With Official Data For General Drills
# ==================================

# ----------------------------------
# General Drills That All Do
# ----------------------------------

general_drills = ["ten_yd_split", "forty", "vertical", "broad_jump", "three_cone", "short_shuttle", "bench_reps"]

# ----------------------------------
# Create Table of Missingness For All General Drills
# ----------------------------------

num_rows = combine_results_df.height

drill_summary_df = pl.DataFrame(
    {
        "drill": general_drills,
        "num_not_missing": [
            combine_results_df[col].count()
            for col in general_drills
        ],
        "perc_not_missing": [
            round(combine_results_df[col].count() / num_rows * 100)
            for col in general_drills
        ],
    }
)

drill_summary_df = (
    drill_summary_df
    .sort("num_not_missing", descending = True)
)

print("Missingness of Official Drill Measurements:")
drill_summary_df

<br><br/>
#### Among Players of Interest, How Many Had Data For Each Drill?

In [ ]:
# =====================================
# Identify Drills For Position of Interest
# =====================================

all_drills = (
    sorted(
        combine_tracking_df
        .select("drill_name")
        .unique()
        .to_series()
        .to_list()
    )
)

print("Position-Specific Drills:")
print(f"    - N = {len(all_drills)}")
all_drills

In [ ]:
# ==================================
# Find Percentage of Players of Interest
# With Tracking Data For Positional Drills
# ==================================

drill_tracking_summary_df = (
    combine_tracking_df
    .group_by("drill_name")
    .agg(
        num_w_atl_1_attempt = pl.col("nfl_id").n_unique()
    )
    .with_columns(
        perc_w_atl_1_attempt = (
            pl.col("num_w_atl_1_attempt")
            / combine_tracking_df.select(pl.col("nfl_id").n_unique()).item()
            * 100
        )
        .round(0)
    )
    .rename({"drill_name": "drill"})
    .sort("num_w_atl_1_attempt", descending = True)
)

print("Missingness in Position-Specific Drills:")
print(f"  - Drills With At Least 80% Participation (N = {drill_tracking_summary_df.filter(pl.col("perc_w_atl_1_attempt") >= 80).height}): {drill_tracking_summary_df.filter(pl.col("perc_w_atl_1_attempt") >= 80).select("drill").to_series().to_list()}")
drill_tracking_summary_df.head(10)

<br><br/>
## In-Game General Exposure
***

#### Among Players of Interest, Identify How Many Snaps They Played


In [ ]:
# =======================================
# Summarize Total Snaps Played
# =======================================

num_with_0_career_snaps = player_exposure_df.filter(pl.col("career_total_snaps") == 0).height
perc_with_0_career_snaps = round(100 * num_with_0_career_snaps / player_exposure_df.height)
print(f"  - Number of Players with      0 Career Snaps: {num_with_0_career_snaps} ({perc_with_0_career_snaps}%)")

num_with_atl_100_career_snaps = player_exposure_df.filter(pl.col("career_total_snaps") >= 100).height
perc_with_atl_100_career_snaps = round(100 * num_with_atl_100_career_snaps / player_exposure_df.height)
print(f"  - Number of Players with >= 100 Career Snaps: {num_with_atl_100_career_snaps} ({perc_with_atl_100_career_snaps}%)")

fig = px.box(
    player_exposure_df, 
    y = "career_total_snaps", 
    points = "all",
    title = "Total Career Snaps For Players of Interest"
)

fig.show()

<br><br/>
#### Among Players of Interest, How Many Have In-Game Tracking Data?

In [ ]:
# =======================================
# Count How Many Player-Plays Worth of 
# Tracking Data Each Player Had
# =======================================

# ---------------------------------------
# Count Number of Player-Plays and Seconds
# Played Among Players of Interest
# ---------------------------------------

player_play_tracking_counts = (
    game_tracking_df
    .with_columns(
        game_play_id = (
            pl.col("game_id").cast(pl.String) +
            "_" +
            pl.col("play_id").cast(pl.String)
        )
    )
    .group_by("nfl_id")
    .agg(
        num_player_plays = pl.col("game_play_id").n_unique(), 
        num_min_played = pl.len() / 10 / 60
    )
)

# ---------------------------------------
# Add Players With No Tracking Data
# ---------------------------------------

player_play_tracking_counts = (
    players_df.select("nfl_id", "display_name")
    .join(
        player_play_tracking_counts, 
        on = "nfl_id", 
        how = "left"
    )
    .with_columns(
        num_player_plays = pl.col("num_player_plays").fill_null(0), 
        num_min_played = pl.col("num_min_played").fill_null(0)
    )
    .sort("num_player_plays", descending = True)
)


print("In-Game Exposure Data:")
player_play_tracking_counts

In [ ]:
# =====================================
# Examine Distribution of Player-Plays
# =====================================

num_with_0_player_plays = player_play_tracking_counts.filter(pl.col("num_player_plays") == 0).height
perc_with_0_player_plays = round(100 * num_with_0_player_plays / player_play_tracking_counts.height)

num_with_atl_100_player_plays = player_play_tracking_counts.filter(pl.col("num_player_plays") >= 100).height
perc_with_atl_100_player_plays = round(100 * num_with_atl_100_player_plays / player_play_tracking_counts.height)

print(f"    - Number of Players With      0 Player-Plays: {num_with_0_player_plays} ({perc_with_0_player_plays}%)")
print(f"    - Number of Players With >= 100 Player-Plays: {num_with_atl_100_player_plays} ({perc_with_atl_100_player_plays}%)")

fig = px.box(
    player_play_tracking_counts, 
    y = "num_player_plays", 
    points = "all", 
    title = "Number of Player-Plays From Players of Interest"
)

fig.show()

In [ ]:
# =====================================
# Examine Distribution of Player-Plays
# =====================================

fig = px.box(
    player_play_tracking_counts, 
    y = "num_min_played", 
    points = "all", 
    title = "Number of Minutes Played From Players of Interest"
)

fig.show()